# General Health Query Chatbot
**Task 4 — Prompt Engineering Based LLM Chatbot**

---

## Problem Statement

People often have basic health questions — *what causes a sore throat, is this medicine safe for children, what does a high fever mean* — but searching online returns overwhelming, jargon-heavy, or unreliable results.

**The problem:** General health information is hard to access in a clear, conversational, and safe way.

**The goal:** Build a chatbot that:
- Answers general health questions in plain, friendly language
- Remembers the conversation so follow-up questions make sense
- Blocks dangerous queries before they reach the AI
- Handles sensitive topics like mental health with extra care
- Always reminds the user it is **not** a substitute for a real doctor

---

In [1]:
%pip install -q -U google-genai
%pip install -q python-dotenv
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv
load_dotenv()

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


False

In [2]:
#dangerous filter
DANGEROUS_KEYWORDS = ["overdose", "how much to kill", "suicide", "self harm","poison myself", "lethal dose", "die", "end my life","kill", "want to die"]
def is_query_safe(user_input):
    lowered = user_input.lower()
    for keyword in DANGEROUS_KEYWORDS:
        if keyword in lowered:
            return False
    return True

In [3]:
#prompt engineering
prompt = """
You are a friendly health information assistant named Hana.
You help people understand general health topics in plain, empathetic language.

Important rules:
- Always recommend seeing a real doctor for personal health decisions
- Never diagnose illnesses or prescribe treatments for a specific person
- If someone seems in distress, gently suggest they call a healthcare provider
- Keep answers concise (3-5 sentences) unless the user asks for more detail
- End every response with: "Remember: this is general info, not medical advice."
"""

In [4]:
#main chatbot
#call API
client = genai.Client()
config = types.GenerateContentConfig(
temperature=0.5,
system_instruction = prompt )
chat = client.chats.create(model='gemini-3.5-flash', config=config)

def ask(question):
    #safety check
    if not is_query_safe(question):
        return "I'm not able to help with that query. If you're in distress, please contact a healthcare provider or call a crisis line."

    response = chat.send_message(question)
    reply = response.text
    return reply
    
print("Health Assistant (type 'quit' to exit)\n")
#chat loop
while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "quit":
        break
    if not user_input:
        continue

    res = ask(user_input)
    print(f"\nGemma: {res}\n")

ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

---

## Results & Final Insights

### What worked well

**Prompt engineering shapes output quality significantly**
Without a system prompt the model gives generic answers. With the persona and rules injected, responses are consistently warm, concise, and always end with a disclaimer.

**Two-tier safety handles edge cases better than a single blocklist**
It distinguishes between *asking about* a sensitive topic vs. *expressing distress*, and handles each appropriately without over-blocking legitimate questions.

**Conversation history makes multi-turn chat feel natural**
Follow-up questions like *'is that safe for kids?'* now work correctly because the model has context of the previous message.

### Example Results

| Query | Safety tier | Outcome |
|---|---|---|
| *'What causes a sore throat?'* | Safe | Clear, friendly explanation |
| *'Is paracetamol safe for children?'* | Safe | Accurate with strong doctor nudge |
| *'I've been feeling really hopeless'* | Sensitive | Empathetic, professional help suggested |
| *'How much paracetamol is a lethal dose?'* | Hard block | Compassionate redirect, no info given |
| *'Tell me more about that'* (follow-up) | Safe | Correctly linked to previous topic |

### Limitations

- **Keyword filters can be bypassed** — a user could rephrase a dangerous query to avoid matching keywords. A second LLM call to classify intent would be more robust.
- **No memory between sessions** — history resets when the notebook restarts. Saving to a JSON file would give persistent memory.
- **Model can be overconfident** — LLMs sometimes state things with more certainty than warranted. The disclaimer at the end of every reply helps, but verified medical sources (RAG) would improve reliability.

### Key Takeaway

> Prompt engineering is not just about asking a question — it is about designing the context, constraints, and persona *around* that question. A well-engineered prompt can turn a general-purpose LLM into a focused, safe domain-specific assistant without any model training at all.

---
**Disclaimer:** This chatbot provides general health information only. It is not a substitute for professional medical advice, diagnosis, or treatment.